In [0]:
# ── List ไฟล์ใน batch-1 เพื่อหาชื่อจริง ──
files = dbutils.fs.ls("/Volumes/bigdata/default/transaction/batch-1/")
for f in files[:5]:
    print(f.path)

dbfs:/Volumes/bigdata/default/transaction/batch-1/test_tx.parquet
dbfs:/Volumes/bigdata/default/transaction/batch-1/train_tx.parquet
dbfs:/Volumes/bigdata/default/transaction/batch-1/val_tx.parquet


In [0]:
from pyspark.sql import functions as F

BASE_PATH = "/Volumes/bigdata/default/transaction"

# ── Step 1: หาไฟล์ .parquet ไฟล์แรกใน batch-1 แบบ dynamic ──
batch1_files = dbutils.fs.ls(f"{BASE_PATH}/batch-1/")
first_parquet = next(f.path for f in batch1_files if f.path.endswith(".parquet"))
print(f"ใช้ไฟล์อ้างอิง schema: {first_parquet}")

# ── Step 2: ดึง schema จากไฟล์นั้น ──
schema = spark.read.parquet(first_parquet).schema
print(schema)

# ── Step 3: โหลดทุก batch พร้อมกัน ──
df = (
    spark.read
    .schema(schema)
    .option("recursiveFileLookup", "true")
    .parquet(BASE_PATH)
)

# ── Step 4: Clean ──
df = (
    df
    .withColumn("transaction_timestamp", F.to_timestamp("transaction_timestamp"))
    .filter(F.col("account_id").isNotNull())
    .filter(F.col("transaction_timestamp").isNotNull())
    .filter(F.col("amount").isNotNull())
    .filter(F.col("txn_type").isNotNull())
)

print(f"Total rows : {df.count():,}")
print(f"Columns    : {df.columns}")
display(df.limit(5))

ใช้ไฟล์อ้างอิง schema: dbfs:/Volumes/bigdata/default/transaction/batch-1/test_tx.parquet
StructType([StructField('transaction_id', StringType(), True), StructField('account_id', StringType(), True), StructField('transaction_timestamp', StringType(), True), StructField('mcc_code', LongType(), True), StructField('channel', StringType(), True), StructField('amount', DoubleType(), True), StructField('txn_type', StringType(), True), StructField('counterparty_id', StringType(), True)])
Total rows : 237,808,547
Columns    : ['transaction_id', 'account_id', 'transaction_timestamp', 'mcc_code', 'channel', 'amount', 'txn_type', 'counterparty_id']


transaction_id,account_id,transaction_timestamp,mcc_code,channel,amount,txn_type,counterparty_id
TXN_0062366125,ACCT_031764,2024-11-14T20:07:15.000Z,2557,UPC,39.39,C,CP_057396
TXN_0062366126,ACCT_031764,2024-11-14T22:14:51.000Z,5651,UPC,1000.0,D,CP_078743
TXN_0062366127,ACCT_031764,2024-11-14T23:25:14.000Z,5682,UPC,165.8,D,CP_097191
TXN_0062366128,ACCT_031764,2024-11-15T05:38:27.000Z,5905,UPD,119118.79,C,CP_015953
TXN_0062366129,ACCT_031764,2024-11-15T09:39:59.000Z,9355,P2A,7280.13,D,CP_040863


In [0]:
# ── ตรวจสอบจำนวนไฟล์แต่ละ batch ──
for batch in ["batch-1", "batch-2", "batch-3", "batch-4"]:
    try:
        files = dbutils.fs.ls(f"{BASE_PATH}/{batch}/")
        parquets = [f for f in files if f.path.endswith(".parquet")]
        print(f"{batch}: {len(parquets)} files")
    except Exception as e:
        print(f"{batch}: ไม่พบ — {e}")

batch-1: 3 files
batch-2: 3 files
batch-3: 3 files
batch-4: 3 files


In [0]:
# ── ดูตัวอย่างข้อมูลใน test_tx.parquet ──
df_test = spark.read.parquet("/Volumes/bigdata/default/transaction/batch-1/test_tx.parquet")

print(f"จำนวน rows : {df_test.count():,}")
print(f"จำนวน cols : {len(df_test.columns)}")
print(f"Columns    : {df_test.columns}")
print("\nSchema:")
df_test.printSchema()

display(df_test.limit(20))

จำนวน rows : 9,239,894
จำนวน cols : 8
Columns    : ['transaction_id', 'account_id', 'transaction_timestamp', 'mcc_code', 'channel', 'amount', 'txn_type', 'counterparty_id']

Schema:
root
 |-- transaction_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- transaction_timestamp: string (nullable = true)
 |-- mcc_code: long (nullable = true)
 |-- channel: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- counterparty_id: string (nullable = true)



transaction_id,account_id,transaction_timestamp,mcc_code,channel,amount,txn_type,counterparty_id
TXN_0062415115,ACCT_031786,2021-12-25T20:45:29,9384,UPD,100.0,D,CP_047952
TXN_0062415116,ACCT_031786,2021-12-27T15:01:39,5651,CHQ,344.97,D,CP_096328
TXN_0062415117,ACCT_031786,2021-12-27T21:02:44,2201,UPD,18000.0,D,CP_051313
TXN_0062415118,ACCT_031786,2021-12-28T16:43:31,5905,UPC,20552.04,D,CP_004090
TXN_0062415119,ACCT_031786,2021-12-28T16:44:15,9355,UPD,58000.0,D,CP_014730
TXN_0062415120,ACCT_031786,2021-12-28T18:56:47,5682,FTC,2.33,C,CP_096328
TXN_0062415121,ACCT_031786,2021-12-29T21:05:45,2769,UPD,183389.23,D,CP_000657
TXN_0062415122,ACCT_031786,2021-12-30T01:44:07,1139,UPC,395500.0,C,CP_006583
TXN_0062415123,ACCT_031786,2021-12-30T09:51:03,5651,UPD,1000.0,D,CP_028698
TXN_0062415124,ACCT_031786,2021-12-31T12:05:36,5651,P2A,75.36,C,CP_019080


In [0]:
# ============================================================
# โหลดข้อมูล 3 ตารางจาก Unity Catalog: bigdata.default
# ============================================================

# --- Test Features ---
test_df = spark.table("bigdata.default.mule_test_features")
print("=== mule_test_features ===")
print(f"Rows : {test_df.count():,}")
test_df.printSchema()
display(test_df.limit(5))

# --- Training Features ---
train_df = spark.table("bigdata.default.mule_training_features")
print("=== mule_training_features ===")
print(f"Rows : {train_df.count():,}")
train_df.printSchema()
display(train_df.limit(5))

# --- Validation Features ---
val_df = spark.table("bigdata.default.mule_val_features")
print("=== mule_val_features ===")
print(f"Rows : {val_df.count():,}")
val_df.printSchema()
display(val_df.limit(20))

=== mule_test_features ===
Rows : 14,414
root
 |-- account_id: string (nullable = true)
 |-- mule_flag_date: date (nullable = true)
 |-- alert_reason: string (nullable = true)
 |-- flagged_by_branch: double (nullable = true)
 |-- is_mule: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- address_last_update_date: string (nullable = true)
 |-- passbook_last_update_date: string (nullable = true)
 |-- joint_account_flag: string (nullable = true)
 |-- nri_flag: string (nullable = true)
 |-- account_status: string (nullable = true)
 |-- product_code: long (nullable = true)
 |-- currency_code: long (nullable = true)
 |-- account_opening_date: string (nullable = true)
 |-- branch_code: long (nullable = true)
 |-- branch_pin: double (nullable = true)
 |-- avg_balance: double (nullable = true)
 |-- product_family: string (nullable = true)
 |-- nomination_flag: string (nullable = true)
 |-- cheque_allowed: string (nullable = true)
 |-- cheque_availed: string (nullable = true)


account_id,mule_flag_date,alert_reason,flagged_by_branch,is_mule,gender,address_last_update_date,passbook_last_update_date,joint_account_flag,nri_flag,account_status,product_code,currency_code,account_opening_date,branch_code,branch_pin,avg_balance,product_family,nomination_flag,cheque_allowed,cheque_availed,num_chequebooks,last_mobile_update_date,kyc_compliant,last_kyc_date,rural_branch,monthly_avg_balance,quarterly_avg_balance,daily_avg_balance,freeze_date,unfreeze_date,total_in,total_out,fast_hr_count,unique_cp_in,unique_cp_out,channel_diversity,mcc_diversity,night_txn_count,balance_ratio,fast_hr_pct,in_out_cp_ratio,round_amount_pct,micro_txn_pct,night_txn_pct,fast_day_count,max_txn_per_day,fast_day_pct
ACCT_002520,null,null,null,0,F,2023-08-28,null,N,N,active,106,1,2025-04-25,6360,743014.0,613.02,S,Y,Y,N,0,null,Y,2025-04-12,Y,684.81,696.43,509.34,null,null,6.369718012000012E7,7.591292252999994E7,2206,34,30,35,58,295,1.1917783862171982,62.83,1.1333333333333333,18.98,3.31,4.52,48,170,57.14
ACCT_060612,null,null,null,0,M,2020-08-28,2021-03-12,N,N,active,106,1,2023-05-11,6250,400169.0,81713.13,S,N,Y,N,0,null,Y,2024-02-20,N,76047.91,68487.95,86256.05,null,null,3.822517104999996E7,4.704359658000001E7,198,25,24,35,58,170,1.2306968232650999,8.97,1.0416666666666667,18.58,3.04,4.1,407,15,53.27
ACCT_007456,null,null,null,0,M,2016-07-09,2023-05-13,N,N,active,187,1,2023-12-07,7587,null,null,S,N,Y,N,0,null,Y,2022-12-01,N,null,10294.4,7586.2,null,null,3.837727383000005E7,1.0102882886999997E8,116,13,12,35,53,126,2.632517080747521,8.15,1.0833333333333333,18.76,3.47,4.6,306,12,55.43
ACCT_018746,null,null,null,0,M,2022-10-22,2022-09-14,N,N,active,100,1,2023-11-20,5942,null,417169.33,S,Y,Y,Y,1,null,Y,2023-11-13,N,374887.6,381042.24,484282.24,null,null,1.946660302000002E7,2.2089701310000014E7,61,25,19,34,54,96,1.1347486403922153,5.34,1.3157894736842106,18.0,3.68,4.42,290,11,51.69
ACCT_005112,null,null,null,0,F,2022-01-29,null,N,N,active,1102,1,2024-07-02,5371,180033.0,2541.2,K,Y,Y,N,0,null,Y,2022-07-19,N,2797.09,3272.67,3370.57,null,null,3.431428702999999E7,1.0843484005000013E8,296,9,8,35,56,169,3.160049339075549,15.5,1.125,18.97,3.01,4.75,211,23,59.1


=== mule_training_features ===
Rows : 67,263
root
 |-- account_id: string (nullable = true)
 |-- mule_flag_date: date (nullable = true)
 |-- alert_reason: string (nullable = true)
 |-- flagged_by_branch: double (nullable = true)
 |-- is_mule: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- address_last_update_date: string (nullable = true)
 |-- passbook_last_update_date: string (nullable = true)
 |-- joint_account_flag: string (nullable = true)
 |-- nri_flag: string (nullable = true)
 |-- account_status: string (nullable = true)
 |-- product_code: long (nullable = true)
 |-- currency_code: long (nullable = true)
 |-- account_opening_date: string (nullable = true)
 |-- branch_code: long (nullable = true)
 |-- branch_pin: double (nullable = true)
 |-- avg_balance: double (nullable = true)
 |-- product_family: string (nullable = true)
 |-- nomination_flag: string (nullable = true)
 |-- cheque_allowed: string (nullable = true)
 |-- cheque_availed: string (nullable = tr

account_id,mule_flag_date,alert_reason,flagged_by_branch,is_mule,gender,address_last_update_date,passbook_last_update_date,joint_account_flag,nri_flag,account_status,product_code,currency_code,account_opening_date,branch_code,branch_pin,avg_balance,product_family,nomination_flag,cheque_allowed,cheque_availed,num_chequebooks,last_mobile_update_date,kyc_compliant,last_kyc_date,rural_branch,monthly_avg_balance,quarterly_avg_balance,daily_avg_balance,freeze_date,unfreeze_date,total_in,total_out,fast_hr_count,unique_cp_in,unique_cp_out,channel_diversity,mcc_diversity,night_txn_count,balance_ratio,fast_hr_pct,in_out_cp_ratio,round_amount_pct,micro_txn_pct,night_txn_pct,fast_day_count,max_txn_per_day,fast_day_pct
ACCT_008595,null,null,null,0,M,2022-05-02,null,N,N,active,101,1,2023-08-31,3201,834019.0,65971.62,S,N,Y,Y,3,null,Y,2023-08-25,N,58771.67,78184.05,71006.56,null,null,6.0491482200000025E7,6.194957704999991E7,223,18,17,35,58,165,1.0241041349454632,10.11,1.0588235294117647,18.49,3.27,4.06,363,16,55.0
ACCT_036515,null,null,null,0,M,2016-02-18,null,N,N,active,1045,1,2022-04-23,1845,431031.0,-65971.88,O,Y,N,N,0,null,Y,2022-07-27,Y,-56808.49,-67849.73,-45901.82,null,null,8226389.970000002,8392457.629999999,12,8,9,33,41,36,1.0201871854611333,2.75,0.8888888888888888,16.6,3.59,4.46,307,5,54.53
ACCT_039148,null,null,null,0,M,2023-03-02,2024-07-24,N,N,active,100,1,2018-04-14,2335,560039.0,33606.58,S,Y,Y,N,0,null,Y,2024-05-15,N,38423.94,30645.31,43846.07,null,null,9094480.83,8385926.069999998,4,10,9,33,46,42,0.9220895867235556,0.95,1.1111111111111112,17.84,3.76,5.1,334,4,50.45
ACCT_063039,null,null,null,0,F,2020-07-16,2025-01-31,N,N,active,193,1,2022-03-18,3826,143037.0,4538.02,K,Y,Y,N,0,2025-06-29,Y,2022-11-14,N,4910.54,5870.55,4234.21,null,null,5.6763919459999986E7,5.0402158930000044E7,166,36,28,35,58,218,0.8879259820230895,6.16,1.2857142857142858,18.73,3.35,4.35,645,13,55.32
ACCT_068595,null,null,null,0,M,2016-09-08,2023-12-21,N,N,active,1133,1,2023-07-31,6541,null,-41981.15,O,N,N,N,0,null,N,2021-06-20,N,-47006.15,-48803.3,-59377.71,null,null,2.2098226899999995E7,4.6198788540000014E7,55,11,12,35,51,88,2.090610651662737,4.93,0.9166666666666666,17.54,3.38,4.25,369,10,57.12


=== mule_val_features ===
Rows : 14,414
root
 |-- account_id: string (nullable = true)
 |-- mule_flag_date: date (nullable = true)
 |-- alert_reason: string (nullable = true)
 |-- flagged_by_branch: double (nullable = true)
 |-- is_mule: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- address_last_update_date: string (nullable = true)
 |-- passbook_last_update_date: string (nullable = true)
 |-- joint_account_flag: string (nullable = true)
 |-- nri_flag: string (nullable = true)
 |-- account_status: string (nullable = true)
 |-- product_code: long (nullable = true)
 |-- currency_code: long (nullable = true)
 |-- account_opening_date: string (nullable = true)
 |-- branch_code: long (nullable = true)
 |-- branch_pin: double (nullable = true)
 |-- avg_balance: double (nullable = true)
 |-- product_family: string (nullable = true)
 |-- nomination_flag: string (nullable = true)
 |-- cheque_allowed: string (nullable = true)
 |-- cheque_availed: string (nullable = true)
 

account_id,mule_flag_date,alert_reason,flagged_by_branch,is_mule,gender,address_last_update_date,passbook_last_update_date,joint_account_flag,nri_flag,account_status,product_code,currency_code,account_opening_date,branch_code,branch_pin,avg_balance,product_family,nomination_flag,cheque_allowed,cheque_availed,num_chequebooks,last_mobile_update_date,kyc_compliant,last_kyc_date,rural_branch,monthly_avg_balance,quarterly_avg_balance,daily_avg_balance,freeze_date,unfreeze_date,total_in,total_out,fast_hr_count,unique_cp_in,unique_cp_out,channel_diversity,mcc_diversity,night_txn_count,balance_ratio,fast_hr_pct,in_out_cp_ratio,round_amount_pct,micro_txn_pct,night_txn_pct,fast_day_count,max_txn_per_day,fast_day_pct
ACCT_068259,null,null,null,0,F,2021-03-13,null,N,N,active,161,1,2022-07-17,1474,201302.0,2638.26,S,N,Y,Y,3,null,Y,2023-09-21,N,2908.16,2025.63,1522.06,null,null,2271397.4200000013,4694782.3900000015,4,13,12,30,37,20,2.0669136755469233,1.58,1.0833333333333333,15.99,10.73,4.05,206,4,52.42
ACCT_034168,null,null,null,0,M,2019-05-11,2022-07-01,N,N,active,187,1,2022-02-08,4751,171016.0,1270.47,S,N,Y,N,0,null,Y,2021-07-19,N,1211.22,1492.68,1411.88,null,null,2.0595833329999994E7,4.323169690000002E7,48,6,5,35,56,90,2.099050628703065,3.82,1.2,17.41,5.35,3.79,550,8,53.5
ACCT_170925,null,null,null,0,M,2020-05-18,2022-10-08,N,N,active,247,1,2023-03-26,5480,639009.0,1713.29,S,N,Y,Y,1,null,Y,2024-07-12,N,1931.96,1803.27,2126.1,null,null,6.854732716000001E7,5.540702355999997E7,107,5,4,34,56,162,0.80830319511469,6.28,1.25,19.07,3.9,5.14,432,13,55.03
ACCT_142572,2024-04-15,Rapid Movement of Funds,8650.0,1,M,2025-04-30,2023-04-10,N,N,frozen,804,1,2024-04-18,8650,360037.0,171307.58,O,Y,N,N,0,2025-01-22,Y,2025-06-10,N,193301.95,198066.87,156926.74,2024-12-23,null,5099982.91,5345285.230000002,8,42,40,31,42,18,1.0480986552952982,3.02,1.05,16.96,3.94,3.55,120,8,56.87
ACCT_168841,null,null,null,0,F,2021-06-22,null,N,N,active,161,1,2022-07-03,4963,120009.0,350.2,S,Y,Y,N,0,null,Y,2021-12-09,N,350.39,386.5,193.65,null,null,3.038945338E7,2.7221980339999996E7,35,28,27,34,53,90,0.8957706477838594,3.0,1.037037037037037,18.92,3.94,4.12,511,8,55.6
ACCT_082054,null,null,null,0,F,2024-11-20,2024-12-25,Y,N,active,169,1,2023-09-05,6025,600063.0,17591.57,S,N,Y,N,0,null,Y,2025-04-04,N,20956.36,14257.9,21769.44,null,null,1.4364976129999999E7,2.912006980999999E7,44,5,4,35,54,71,2.0271575494779466,4.75,1.25,19.26,3.04,4.07,323,10,54.38
ACCT_186669,null,null,null,0,F,2020-01-27,2023-01-26,N,N,active,193,1,2021-06-19,6721,null,1933.71,K,Y,Y,N,0,null,Y,2021-06-26,N,1981.3,2106.95,2300.92,null,null,1.1711122359999998E7,1.1836056209999992E7,11,4,3,32,51,61,1.010667965559536,1.62,1.3333333333333333,18.99,7.23,4.46,450,6,51.14
ACCT_029350,null,null,null,0,M,2022-05-25,2024-11-06,N,N,active,100,1,2018-12-14,5794,400185.0,null,S,Y,Y,N,0,null,Y,2021-10-22,N,null,3657.97,4345.05,null,null,2.3206805799999986E7,1.922914764999999E7,14,17,16,33,51,67,0.8285994986005356,1.46,1.0625,18.81,3.58,3.63,597,6,52.88
ACCT_046337,null,null,null,0,M,2019-12-31,2021-06-18,N,N,active,247,1,2023-04-01,1787,411032.0,0.0,S,Y,Y,Y,1,null,Y,2024-03-04,N,0.0,0.0,0.0,null,null,4.1266777870000005E7,1.441615961999999E7,30,23,29,33,54,78,0.3493405679845968,3.22,0.7931034482758621,20.18,3.1,4.64,393,10,54.66
ACCT_125165,null,null,null,0,M,2020-09-24,2024-07-25,N,Y,active,100,1,2019-10-10,8295,600127.0,3844.24,S,Y,Y,Y,3,2025-01-27,Y,2023-02-03,N,3648.35,3188.8,3162.14,null,null,1.936857406000002E7,1.2625454119999997E7,12,10,9,33,50,65,0.6518525360147231,1.77,1.1111111111111112,16.47,4.48,4.7,486,6,51.21


ตรวจสอบ ID ถูก Split ดีไหม มีID ซ้ำไหม Train ID transaction ตรงกับ Train ID mule table ไหม 

In [0]:
from pyspark.sql import functions as F

# ============================================================
# STEP 1: โหลด Transaction ทุก Batch (batch-1, 2, 3)
# ============================================================

schema = spark.read.parquet(
    "/Volumes/workspace/default/mule_raw/transactions/batch-3/part_0297.parquet"
).schema

txn_all = spark.read \
    .schema(schema) \
    .option("recursiveFileLookup", "true") \
    .parquet("/Volumes/workspace/default/mule_raw/transactions/")

# ดึงเฉพาะ account_id ที่ unique จาก transactions
txn_accounts = txn_all.select("account_id").distinct()

print(f"Unique account_id จาก Transactions (ทุก batch) : {txn_accounts.count():,}")

# ============================================================
# STEP 2: โหลด mule_training_features จาก Unity Catalog
# ============================================================

train_df = spark.table("bigdata.default.mule_training_features")

# ดึงเฉพาะ account_id ที่ unique จาก train
train_accounts = train_df.select("account_id").distinct()

print(f"Unique account_id จาก mule_training_features      : {train_accounts.count():,}")

# ============================================================
# STEP 3: Left Join — Batch ← Mule Train
# ============================================================

joined = txn_accounts.join(
    train_accounts.withColumn("in_mule_train", F.lit(True)),
    on="account_id",
    how="left"
)

# ============================================================
# STEP 4: วิเคราะห์ผล
# ============================================================

matched     = joined.filter(F.col("in_mule_train") == True)
not_matched = joined.filter(F.col("in_mule_train").isNull())

total_txn   = txn_accounts.count()
total_match = matched.count()
total_miss  = not_matched.count()

print("\n" + "="*50)
print("  ผลการตรวจสอบ account_id")
print("="*50)
print(f"  account_id ใน Transactions (ทุก batch) : {total_txn:,}")
print(f"  ✅ พบใน mule_training_features          : {total_match:,}")
print(f"  ❌ ไม่พบใน mule_training_features       : {total_miss:,}")
print(f"  Coverage                                : {total_match/total_txn*100:.2f}%")
print("="*50)

# ============================================================
# STEP 5: แสดง account_id ที่ไม่ตรงกัน (ถ้ามี)
# ============================================================

if total_miss > 0:
    print(f"\n⚠️  ตัวอย่าง account_id ที่อยู่ใน Batch แต่ไม่อยู่ใน mule_training_features:")
    display(not_matched.limit(20))
else:
    print("\n✅ ทุก account_id ใน Transactions ตรงกับ mule_training_features ครบถ้วน!")

# ============================================================
# STEP 6: ตรวจสอบย้อนกลับ — มี ID ใน Mule Train ที่ไม่มี Txn?
# ============================================================

reverse_join = train_accounts.join(
    txn_accounts.withColumn("in_txn", F.lit(True)),
    on="account_id",
    how="left"
)

orphan_mule = reverse_join.filter(F.col("in_txn").isNull())
print(f"\n🔍 account_id ที่อยู่ใน mule_training_features แต่ไม่มี Transaction : {orphan_mule.count():,}")
if orphan_mule.count() > 0:
    display(orphan_mule.limit(10))

In [0]:
from pyspark.sql import functions as F

BASE_PATH = "/Volumes/bigdata/default/transaction"

# ============================================================
# STEP 1: โหลด train_tx.parquet จากทุก batch (1-4)
# ============================================================

train_batches = []
for batch in ["batch-1", "batch-2", "batch-3", "batch-4"]:
    path = f"{BASE_PATH}/{batch}/train_tx.parquet"
    try:
        df_batch = spark.read.parquet(path)
        df_batch = df_batch.withColumn("source_batch", F.lit(batch))
        train_batches.append(df_batch)
        print(f"✅ {batch}/train_tx.parquet → {df_batch.count():,} rows")
    except Exception as e:
        print(f"❌ {batch}: ไม่พบ — {e}")

# Union ทุก batch เข้าด้วยกัน
from functools import reduce
txn_train_all = reduce(lambda a, b: a.union(b), train_batches)
print(f"\n📦 Total rows (ทุก batch รวม) : {txn_train_all.count():,}")

# ============================================================
# STEP 2: ดึง unique account_id จาก Batch Train
# ============================================================

txn_accounts = txn_train_all.select("account_id").distinct()
print(f"🔑 Unique account_id จาก Batch train ทั้งหมด : {txn_accounts.count():,}")

# ============================================================
# STEP 3: ดึง unique account_id จาก mule_training_features
# ============================================================

mule_accounts = train_df.select("account_id").distinct()
print(f"🔑 Unique account_id จาก mule_training_features : {mule_accounts.count():,}")

# ============================================================
# STEP 4: Left Join — Batch Train ← Mule Train
# ============================================================

joined = txn_accounts.join(
    mule_accounts.withColumn("in_mule_train", F.lit(True)),
    on="account_id",
    how="left"
)

matched     = joined.filter(F.col("in_mule_train") == True)
not_matched = joined.filter(F.col("in_mule_train").isNull())

total     = txn_accounts.count()
n_match   = matched.count()
n_miss    = not_matched.count()

print("\n" + "="*55)
print("   ผลการตรวจสอบ account_id")
print("="*55)
print(f"   Batch train (unique IDs)          : {total:,}")
print(f"   ✅ พบใน mule_training_features    : {n_match:,}")
print(f"   ❌ ไม่พบใน mule_training_features : {n_miss:,}")
print(f"   Coverage                          : {n_match/total*100:.2f}%")
print("="*55)

if n_miss > 0:
    print(f"\n⚠️  ตัวอย่าง account_id ที่ขาดหายไป:")
    display(not_matched.limit(20))
else:
    print("\n✅ ทุก account_id ใน Batch Train ตรงกับ mule_training_features ครบ 100%!")

# ============================================================
# STEP 5: ตรวจสอบย้อนกลับ — มี ID ใน Mule ที่ไม่มี Txn?
# ============================================================

reverse = mule_accounts.join(
    txn_accounts.withColumn("in_txn", F.lit(True)),
    on="account_id",
    how="left"
)

orphan = reverse.filter(F.col("in_txn").isNull())
n_orphan = orphan.count()
print(f"\n🔍 ID ใน mule_training_features แต่ไม่มีใน Batch train : {n_orphan:,}")
if n_orphan > 0:
    display(orphan.limit(10))

✅ batch-1/train_tx.parquet → 42,179,030 rows
✅ batch-2/train_tx.parquet → 42,037,447 rows
✅ batch-3/train_tx.parquet → 42,259,685 rows
✅ batch-4/train_tx.parquet → 40,260,213 rows

📦 Total rows (ทุก batch รวม) : 166,736,375
🔑 Unique account_id จาก Batch train ทั้งหมด : 67,263
🔑 Unique account_id จาก mule_training_features : 67,263

   ผลการตรวจสอบ account_id
   Batch train (unique IDs)          : 67,263
   ✅ พบใน mule_training_features    : 67,263
   ❌ ไม่พบใน mule_training_features : 0
   Coverage                          : 100.00%

✅ ทุก account_id ใน Batch Train ตรงกับ mule_training_features ครบ 100%!

🔍 ID ใน mule_training_features แต่ไม่มีใน Batch train : 0


In [0]:
from pyspark.sql import functions as F
from functools import reduce

BASE_PATH = "/Volumes/bigdata/default/transaction"

# ============================================================
# CONFIG: กำหนด mapping ไฟล์ ↔ mule table
# ============================================================

checks = [
    {
        "label"      : "TRAIN",
        "file_name"  : "train_tx.parquet",
        "mule_df"    : train_df,   # โหลดไว้แล้วจาก cell ก่อน
        "mule_name"  : "mule_training_features"
    },
    {
        "label"      : "VAL",
        "file_name"  : "val_tx.parquet",
        "mule_df"    : val_df,     # โหลดไว้แล้วจาก cell ก่อน
        "mule_name"  : "mule_val_features"
    },
    {
        "label"      : "TEST",
        "file_name"  : "test_tx.parquet",
        "mule_df"    : test_df,    # โหลดไว้แล้วจาก cell ก่อน
        "mule_name"  : "mule_test_features"
    },
]

BATCHES = ["batch-1", "batch-2", "batch-3", "batch-4"]

# ============================================================
# รัน ตรวจสอบทุก set
# ============================================================

summary_rows = []

for chk in checks:
    label     = chk["label"]
    fname     = chk["file_name"]
    mule_df   = chk["mule_df"]
    mule_name = chk["mule_name"]

    print("\n" + "="*60)
    print(f"  🔍 ตรวจสอบ SET: {label}  |  ไฟล์: {fname}")
    print("="*60)

    # --- โหลดทุก batch ---
    batch_dfs = []
    for batch in BATCHES:
        path = f"{BASE_PATH}/{batch}/{fname}"
        try:
            df_b = spark.read.parquet(path).select("account_id")
            batch_dfs.append(df_b)
            print(f"  ✅ {batch}/{fname} → โหลดสำเร็จ")
        except Exception as e:
            print(f"  ❌ {batch}/{fname} → ไม่พบ")

    if not batch_dfs:
        print(f"  ⚠️  ไม่มีไฟล์ใดโหลดได้ ข้าม {label}")
        continue

    # --- Union + Distinct ---
    txn_ids  = reduce(lambda a, b: a.union(b), batch_dfs).distinct()
    mule_ids = mule_df.select("account_id").distinct()

    n_txn  = txn_ids.count()
    n_mule = mule_ids.count()

    # --- Left Join: Batch → Mule ---
    joined = txn_ids.join(
        mule_ids.withColumn("in_mule", F.lit(True)),
        on="account_id",
        how="left"
    )
    matched     = joined.filter(F.col("in_mule") == True).count()
    not_matched = joined.filter(F.col("in_mule").isNull()).count()

    # --- Reverse: Mule → Batch ---
    reverse  = mule_ids.join(
        txn_ids.withColumn("in_txn", F.lit(True)),
        on="account_id",
        how="left"
    )
    orphan = reverse.filter(F.col("in_txn").isNull()).count()

    coverage = matched / n_txn * 100 if n_txn > 0 else 0

    print(f"\n  {'Batch IDs (unique)':<35}: {n_txn:,}")
    print(f"  {mule_name + ' IDs (unique)':<35}: {n_mule:,}")
    print(f"  {'✅ Batch → Mule (ตรงกัน)':<35}: {matched:,}")
    print(f"  {'❌ Batch → Mule (ไม่พบใน Mule)':<35}: {not_matched:,}")
    print(f"  {'🔁 Mule → Batch (Mule ไม่มี Txn)':<35}: {orphan:,}")
    print(f"  {'Coverage':<35}: {coverage:.2f}%")

    # สรุป
    if not_matched == 0 and orphan == 0:
        verdict = "✅ ตรงกันสมบูรณ์ 100%"
    elif not_matched == 0 and orphan > 0:
        verdict = f"⚠️  Batch ครบ แต่ Mule มี {orphan:,} ID ที่ไม่มี Txn"
    elif not_matched > 0 and orphan == 0:
        verdict = f"⚠️  Mule ครบ แต่ Batch มี {not_matched:,} ID ที่ไม่อยู่ใน Mule"
    else:
        verdict = f"❌ ไม่ตรงกัน ({not_matched:,} ขาดใน Mule, {orphan:,} ขาดใน Batch)"

    print(f"\n  📋 สรุป: {verdict}")

    summary_rows.append({
        "Set"         : label,
        "File"        : fname,
        "Mule Table"  : mule_name,
        "Batch IDs"   : n_txn,
        "Mule IDs"    : n_mule,
        "Matched"     : matched,
        "Missing→Mule": not_matched,
        "Missing→Txn" : orphan,
        "Coverage%"   : round(coverage, 2),
        "Verdict"     : verdict,
    })

# ============================================================
# SUMMARY TABLE
# ============================================================

import pandas as pd
summary_pd = pd.DataFrame(summary_rows)

print("\n\n" + "="*60)
print("  📊 SUMMARY ทั้งหมด")
print("="*60)
display(summary_pd)


  🔍 ตรวจสอบ SET: TRAIN  |  ไฟล์: train_tx.parquet
  ✅ batch-1/train_tx.parquet → โหลดสำเร็จ
  ✅ batch-2/train_tx.parquet → โหลดสำเร็จ
  ✅ batch-3/train_tx.parquet → โหลดสำเร็จ
  ✅ batch-4/train_tx.parquet → โหลดสำเร็จ

  Batch IDs (unique)                 : 67,263
  mule_training_features IDs (unique): 67,263
  ✅ Batch → Mule (ตรงกัน)            : 67,263
  ❌ Batch → Mule (ไม่พบใน Mule)      : 0
  🔁 Mule → Batch (Mule ไม่มี Txn)    : 0
  Coverage                           : 100.00%

  📋 สรุป: ✅ ตรงกันสมบูรณ์ 100%

  🔍 ตรวจสอบ SET: VAL  |  ไฟล์: val_tx.parquet
  ✅ batch-1/val_tx.parquet → โหลดสำเร็จ
  ✅ batch-2/val_tx.parquet → โหลดสำเร็จ
  ✅ batch-3/val_tx.parquet → โหลดสำเร็จ
  ✅ batch-4/val_tx.parquet → โหลดสำเร็จ

  Batch IDs (unique)                 : 14,414
  mule_val_features IDs (unique)     : 14,414
  ✅ Batch → Mule (ตรงกัน)            : 14,414
  ❌ Batch → Mule (ไม่พบใน Mule)      : 0
  🔁 Mule → Batch (Mule ไม่มี Txn)    : 0
  Coverage                           : 100.00%

  📋 ส

Set,File,Mule Table,Batch IDs,Mule IDs,Matched,Missing→Mule,Missing→Txn,Coverage%,Verdict
TRAIN,train_tx.parquet,mule_training_features,67263,67263,67263,0,0,100.0,✅ ตรงกันสมบูรณ์ 100%
VAL,val_tx.parquet,mule_val_features,14414,14414,14414,0,0,100.0,✅ ตรงกันสมบูรณ์ 100%
TEST,test_tx.parquet,mule_test_features,14414,14414,14414,0,0,100.0,✅ ตรงกันสมบูรณ์ 100%


In [0]:
from pyspark.sql import functions as F
from functools import reduce

BASE_PATH = "/Volumes/bigdata/default/transaction"
BATCHES   = ["batch-1", "batch-2", "batch-3", "batch-4"]

# ============================================================
# STEP 1: รวม account_id จากทุก batch แยกตาม set
# ============================================================

def load_ids(file_name):
    dfs = []
    for batch in BATCHES:
        path = f"{BASE_PATH}/{batch}/{file_name}"
        try:
            dfs.append(spark.read.parquet(path).select("account_id"))
        except:
            pass
    return reduce(lambda a, b: a.union(b), dfs).distinct() if dfs else None

train_ids = load_ids("train_tx.parquet").withColumn("in_train", F.lit(True))
val_ids   = load_ids("val_tx.parquet").withColumn("in_val",   F.lit(True))
test_ids  = load_ids("test_tx.parquet").withColumn("in_test",  F.lit(True))

n_train = train_ids.count()
n_val   = val_ids.count()
n_test  = test_ids.count()

print(f"Unique IDs → TRAIN : {n_train:,}")
print(f"Unique IDs → VAL   : {n_val:,}")
print(f"Unique IDs → TEST  : {n_test:,}")

# ============================================================
# STEP 2: Full Outer Join ทั้ง 3 set
# ============================================================

all_ids = (
    train_ids
    .join(val_ids,  on="account_id", how="full_outer")
    .join(test_ids, on="account_id", how="full_outer")
    .fillna(False, subset=["in_train", "in_val", "in_test"])
)

# ============================================================
# STEP 3: จำแนก overlap ทุกกรณี
# ============================================================

# ซ้ำ Train ↔ Val
train_val   = all_ids.filter(F.col("in_train") & F.col("in_val")   & ~F.col("in_test"))
# ซ้ำ Train ↔ Test
train_test  = all_ids.filter(F.col("in_train") & F.col("in_test")  & ~F.col("in_val"))
# ซ้ำ Val ↔ Test
val_test    = all_ids.filter(F.col("in_val")   & F.col("in_test")  & ~F.col("in_train"))
# ซ้ำทั้ง 3
all_three   = all_ids.filter(F.col("in_train") & F.col("in_val")   & F.col("in_test"))
# ไม่ซ้ำใคร (เฉพาะตัวเอง)
only_train  = all_ids.filter(F.col("in_train") & ~F.col("in_val")  & ~F.col("in_test"))
only_val    = all_ids.filter(F.col("in_val")   & ~F.col("in_train")& ~F.col("in_test"))
only_test   = all_ids.filter(F.col("in_test")  & ~F.col("in_train")& ~F.col("in_val"))

n_tv  = train_val.count()
n_tt  = train_test.count()
n_vt  = val_test.count()
n_all = all_three.count()
n_ot  = only_train.count()
n_ov  = only_val.count()
n_ox  = only_test.count()

# ============================================================
# STEP 4: สรุปผล
# ============================================================

print("\n" + "="*55)
print("   📊 ผลการตรวจสอบการซ้ำของ account_id")
print("="*55)
print(f"   เฉพาะ TRAIN เท่านั้น           : {n_ot:,}")
print(f"   เฉพาะ VAL เท่านั้น             : {n_ov:,}")
print(f"   เฉพาะ TEST เท่านั้น            : {n_ox:,}")
print("-"*55)
print(f"   ⚠️  ซ้ำ TRAIN ↔ VAL             : {n_tv:,}")
print(f"   ⚠️  ซ้ำ TRAIN ↔ TEST            : {n_tt:,}")
print(f"   ⚠️  ซ้ำ VAL   ↔ TEST            : {n_vt:,}")
print(f"   ❌  ซ้ำทั้ง TRAIN + VAL + TEST  : {n_all:,}")
print("="*55)

total_overlap = n_tv + n_tt + n_vt + n_all
if total_overlap == 0:
    print("\n   ✅ ไม่มีการซ้ำกันเลย — Data Leakage Free!")
else:
    print(f"\n   ❌ พบ Data Leakage รวม {total_overlap:,} IDs")

# ============================================================
# STEP 5: แสดงตัวอย่าง ID ที่ซ้ำ (ถ้ามี)
# ============================================================

for name, df, n in [
    ("TRAIN ↔ VAL",            train_val,  n_tv),
    ("TRAIN ↔ TEST",           train_test, n_tt),
    ("VAL   ↔ TEST",           val_test,   n_vt),
    ("TRAIN + VAL + TEST",     all_three,  n_all),
]:
    if n > 0:
        print(f"\n⚠️  ตัวอย่าง ID ที่ซ้ำ ({name}) — {n:,} IDs:")
        display(df.limit(10))

Unique IDs → TRAIN : 67,263
Unique IDs → VAL   : 14,414
Unique IDs → TEST  : 14,414

   📊 ผลการตรวจสอบการซ้ำของ account_id
   เฉพาะ TRAIN เท่านั้น           : 67,263
   เฉพาะ VAL เท่านั้น             : 14,414
   เฉพาะ TEST เท่านั้น            : 14,414
-------------------------------------------------------
   ⚠️  ซ้ำ TRAIN ↔ VAL             : 0
   ⚠️  ซ้ำ TRAIN ↔ TEST            : 0
   ⚠️  ซ้ำ VAL   ↔ TEST            : 0
   ❌  ซ้ำทั้ง TRAIN + VAL + TEST  : 0

   ✅ ไม่มีการซ้ำกันเลย — Data Leakage Free!


Joint Mule train กับ trasaction train

Feature1 :Rapid